# Remote Work & Employee Wellbeing: Data Analytics Portfolio

## 1. Introduction and Analytical Objective

This notebook documents the data preparation, exploratory analysis, dashboard metric calculations, and statistical testing conducted using the Remote Work & Health Impact Survey 2025 dataset.

The analysis supports a managerial decision about hybrid working arrangements by examining employee burnout, social isolation, reported health information, and salary groups across remote, hybrid, and onsite employees.

The analytical workflow includes:

1. Data loading and quality assessment.
2. Missing-value assessment and preparation of analytical variables.
3. Descriptive analysis and validation of Power BI dashboard metrics.
4. Chi-square test of independence between work arrangement and burnout level.
5. Interpretation of findings and analytical limitations.

The Python calculations provide a reproducible validation of the results presented in the Power BI dashboard.


In [2]:
import pandas as pd

df = pd.read_csv(
    "data/post_pandemic_remote_work_health_impact_2025.csv"
)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (3157, 14)


,Survey_Date,Age,Gender,Region,Industry,Job_Role,Work_Arrangement,Hours_Per_Week,Mental_Health_Status,Burnout_Level,Work_Life_Balance_Score,Physical_Health_Issues,Social_Isolation_Score,Salary_Range
0,2025-06-01,27,Female,Asia,Professional Services,Data Analyst,Onsite,64,Stress Disorder,High,3,Shoulder Pain; Neck Pain,2,$40K-60K
1,2025-06-01,37,Female,Asia,Professional Services,Data Analyst,Onsite,37,Stress Disorder,High,4,Back Pain,2,$80K-100K
2,2025-06-01,32,Female,Africa,Education,Business Analyst,Onsite,36,ADHD,High,3,Shoulder Pain; Eye Strain,2,$80K-100K
3,2025-06-01,40,Female,Europe,Education,Data Analyst,Onsite,63,ADHD,Medium,1,Shoulder Pain; Eye Strain,2,$60K-80K
4,2025-06-01,30,Male,South America,Manufacturing,DevOps Engineer,Hybrid,65,NaN,Medium,5,NaN,4,$60K-80K


In [3]:
# Check data types and missing values

print("Data types:")
print(df.dtypes)

print("\nMissing values:")
missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2)
})

display(missing)

print("Duplicate rows:", df.duplicated().sum())

Data types:
Survey_Date                  str
Age                        int64
Gender                       str
Region                       str
Industry                     str
Job_Role                     str
Work_Arrangement             str
Hours_Per_Week             int64
Mental_Health_Status         str
Burnout_Level                str
Work_Life_Balance_Score    int64
Physical_Health_Issues       str
Social_Isolation_Score     int64
Salary_Range                 str
dtype: object

Missing values:


,Missing Count,Missing %
Survey_Date,0,0.00
Age,0,0.00
Gender,0,0.00
Region,0,0.00
Industry,0,0.00
Job_Role,0,0.00
Work_Arrangement,0,0.00
Hours_Per_Week,0,0.00
Mental_Health_Status,799,25.31
Burnout_Level,0,0.00


Duplicate rows: 0


## 2. Data Quality Assessment and Preparation

The dataset contains 3,157 survey responses and 14 original variables. Data quality checks were performed to examine variable types, missing values, and duplicate records.

### Data Quality Findings

- No fully duplicated records were identified.
- `Mental_Health_Status` contains 799 missing values (25.31%).
- `Physical_Health_Issues` contains 280 missing values (8.87%).
- The remaining 12 variables have no missing values.

### Missing-Value Handling

No records were removed, and missing health information was not replaced with zero. Missing values may represent unreported information rather than an absence of health conditions.

Two binary indicators were created to identify whether information was recorded in each health-related field:

- `Mental_Health_Reported`
- `Physical_Health_Reported`

A value of 1 indicates a non-missing entry, while 0 indicates a missing entry.

These indicators measure the availability of recorded health information, not clinically confirmed health conditions.

The original CSV file was preserved without modification.

In [4]:
# Create indicators for reported health information

df["Mental_Health_Reported"] = (
    df["Mental_Health_Status"].notna()
).astype(int)

df["Physical_Health_Reported"] = (
    df["Physical_Health_Issues"].notna()
).astype(int)

# Check the results
display(
    df[
        [
            "Mental_Health_Status",
            "Mental_Health_Reported",
            "Physical_Health_Issues",
            "Physical_Health_Reported"
        ]
    ].head(10)
)

,Mental_Health_Status,Mental_Health_Reported,Physical_Health_Issues,Physical_Health_Reported
0,Stress Disorder,1,Shoulder Pain; Neck Pain,1
1,Stress Disorder,1,Back Pain,1
2,ADHD,1,Shoulder Pain; Eye Strain,1
3,ADHD,1,Shoulder Pain; Eye Strain,1
4,NaN,0,NaN,0
5,Burnout,1,Back Pain; Shoulder Pain,1
6,NaN,0,Back Pain; Shoulder Pain; Wrist Pain,1
7,Anxiety,1,NaN,0
8,Anxiety,1,Neck Pain,1
9,Burnout,1,Back Pain,1


## 3. Descriptive Analysis and Dashboard Validation

Descriptive analysis was conducted to compare employee burnout and wellbeing across three work arrangements: remote, hybrid, and onsite.

A binary variable, `High_Burnout`, was created to identify respondents reporting a high level of burnout.

### Dashboard Metrics

The following measures were calculated and independently validated against the Power BI dashboard:

1. Total number of respondents and overall high-burnout rate.
2. High-burnout rate by work arrangement.
3. Distribution of low, medium, and high burnout within each work arrangement.
4. Percentage of respondents with recorded physical health information.
5. Average social isolation score by work arrangement.
6. Percentage of respondents with recorded mental health information.
7. High-burnout rate by social isolation score and work arrangement.
8. High-burnout rate by salary range and work arrangement.

All percentage-based burnout measures use the relevant group size as their denominator.

### Analytical Approach

Group comparisons and percentage distributions were used to identify patterns relevant to the managerial decision about hybrid working arrangements.

The Python calculations were used to reproduce and validate the metrics displayed in the two-page Power BI dashboard.

The findings describe associations within the survey sample and do not establish causal relationships.


In [5]:
# Create a binary indicator for high burnout
df["High_Burnout"] = (
    df["Burnout_Level"] == "High"
).astype(int)

# Calculate dashboard KPIs
total_respondents = len(df)

high_burnout_rate = df["High_Burnout"].mean() * 100

burnout_by_arrangement = (
    df.groupby("Work_Arrangement")["High_Burnout"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

highest_burnout_group = burnout_by_arrangement.idxmax()

print("Total Respondents:", total_respondents)
print(f"High Burnout Rate: {high_burnout_rate:.1f}%")
print("Highest Burnout Group:", highest_burnout_group)

display(burnout_by_arrangement.round(2))

Total Respondents: 3157
High Burnout Rate: 33.1%
Highest Burnout Group: Remote


Work_Arrangement
Remote    46.43
Hybrid    35.75
Onsite    26.44
Name: High_Burnout, dtype: float64

In [6]:
# Count employees by work arrangement and burnout level
burnout_counts = pd.crosstab(
    df["Work_Arrangement"],
    df["Burnout_Level"]
)

# Calculate percentages within each work arrangement
burnout_distribution = (
    burnout_counts
    .div(burnout_counts.sum(axis=1), axis=0)
    .mul(100)
)

# Display results in the same order as the dashboard
burnout_distribution = burnout_distribution.reindex(
    index=["Onsite", "Hybrid", "Remote"],
    columns=["Low", "Medium", "High"]
)

display(burnout_distribution.round(2))


Burnout_Level,Low,Medium,High
Work_Arrangement,,,
Onsite,30.15,43.41,26.44
Hybrid,19.17,45.08,35.75
Remote,13.78,39.80,46.43


In [7]:
# Calculate reported physical health issues by work arrangement

physical_health_rates = (
    df.groupby("Work_Arrangement")["Physical_Health_Reported"]
    .mean()
    .mul(100)
    .reindex(["Remote", "Hybrid", "Onsite"])
)

display(
    physical_health_rates
    .round(1)
    .to_frame("Reported Physical Health (%)")
)

,Reported Physical Health (%)
Work_Arrangement,
Remote,92.3
Hybrid,91.0
Onsite,90.8


In [8]:
# Calculate average social isolation score by work arrangement

isolation_by_arrangement = (
    df.groupby("Work_Arrangement")["Social_Isolation_Score"]
    .mean()
    .reindex(["Remote", "Hybrid", "Onsite"])
)

display(
    isolation_by_arrangement
    .round(2)
    .to_frame("Average Social Isolation Score")
)

,Average Social Isolation Score
Work_Arrangement,
Remote,3.50
Hybrid,2.75
Onsite,2.38


In [9]:
# Calculate reported mental health information by work arrangement

mental_health_rates = (
    df.groupby("Work_Arrangement")["Mental_Health_Reported"]
    .mean()
    .mul(100)
    .reindex(["Onsite", "Hybrid", "Remote"])
)

display(
    mental_health_rates
    .round(1)
    .to_frame("Reported Mental Health (%)")
)

,Reported Mental Health (%)
Work_Arrangement,
Onsite,76.4
Hybrid,73.4
Remote,72.3


In [10]:
# High burnout rate by social isolation and work arrangement

burnout_by_isolation = (
    df.groupby(
        ["Social_Isolation_Score", "Work_Arrangement"]
    )["High_Burnout"]
    .mean()
    .mul(100)
    .unstack()
)

burnout_by_isolation = burnout_by_isolation.reindex(
    columns=["Hybrid", "Onsite", "Remote"]
)

display(burnout_by_isolation.round(1))

Work_Arrangement,Hybrid,Onsite,Remote
Social_Isolation_Score,,,
1,38.1,27.8,50.0
2,36.8,28.2,51.7
3,34.8,21.3,48.5
4,32.8,29.9,41.9
5,41.2,28.0,47.1


In [11]:
# High burnout rate by salary range and work arrangement

burnout_by_salary = (
    df.groupby(
        ["Salary_Range", "Work_Arrangement"]
    )["High_Burnout"]
    .mean()
    .mul(100)
    .unstack()
)

# Match the dashboard's row and column order
salary_order = [
    "$40K-60K",
    "$60K-80K",
    "$80K-100K",
    "$100K-120K",
    "$120K+"
]

burnout_by_salary = burnout_by_salary.reindex(
    index=salary_order,
    columns=["Hybrid", "Onsite", "Remote"]
)

display(burnout_by_salary.round(1))

Work_Arrangement,Hybrid,Onsite,Remote
Salary_Range,,,
$40K-60K,39.0,28.9,46.3
$60K-80K,34.6,25.0,43.0
$80K-100K,34.7,27.3,48.6
$100K-120K,37.2,24.8,48.7
$120K+,35.4,28.3,48.7


## 4. Statistical Analysis: Chi-Square Test of Independence

### Objective

A chi-square test of independence was conducted to examine whether work arrangement and burnout level were statistically associated.

### Hypotheses

- **Null hypothesis (H₀):** Work arrangement and burnout level are independent.
- **Alternative hypothesis (H₁):** Work arrangement and burnout level are associated.

### Method

A contingency table was constructed using the three work arrangements (onsite, hybrid, and remote) and three burnout levels (low, medium, and high).

The chi-square test was performed at a significance level of 0.05. Expected cell frequencies were examined to assess the suitability of the test. Cramer's V was calculated to evaluate the strength of the association.

### Results

- Chi-square statistic: 117.85
- Degrees of freedom: 4
- p-value: < 0.001
- Minimum expected cell frequency: 138.76
- Cramer's V: 0.137

### Interpretation

The null hypothesis was rejected, indicating a statistically significant association between work arrangement and burnout level.

However, Cramer's V indicates a small association. Statistical significance should therefore not be interpreted as evidence of a strong relationship.

The results are observational and do not establish that work arrangement causes differences in burnout.

In [12]:
from scipy.stats import chi2_contingency
import numpy as np

# Create a contingency table
contingency_table = pd.crosstab(
    df["Work_Arrangement"],
    df["Burnout_Level"]
).reindex(
    index=["Onsite", "Hybrid", "Remote"],
    columns=["Low", "Medium", "High"]
)

display(contingency_table)

# Perform a Chi-square test
chi2, p_value, dof, expected = chi2_contingency(
    contingency_table
)

# Calculate Cramer's V (effect size)
n = contingency_table.to_numpy().sum()
min_dimension = min(contingency_table.shape) - 1

cramers_v = np.sqrt(
    chi2 / (n * min_dimension)
)

print(f"Chi-square statistic: {chi2:.2f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_value:.6g}")
print(f"Minimum expected frequency: {expected.min():.2f}")
print(f"Cramer's V: {cramers_v:.3f}")

Burnout_Level,Low,Medium,High
Work_Arrangement,,,
Onsite,471,678,413
Hybrid,193,454,360
Remote,81,234,273


Chi-square statistic: 117.85
Degrees of freedom: 4
P-value: 1.53694e-24
Minimum expected frequency: 138.76
Cramer's V: 0.137


## 5. Key Findings and Managerial Implications

### Key Findings

The analysis of 3,157 survey responses identified the following patterns:

1. **Burnout differs across work arrangements.** Remote employees reported the highest high-burnout rate (46.4%), followed by hybrid employees (35.7%) and onsite employees (26.4%).

2. **Remote employees reported greater social isolation.** Their average social isolation score was 3.50, compared with 2.75 for hybrid employees and 2.38 for onsite employees.

3. **The burnout pattern persists across the examined groups.** Remote employees had the highest high-burnout rate at every social isolation score and across all five salary ranges.

4. **Health information requires careful interpretation.** Recorded physical and mental health entries were compared across work arrangements, but missing entries cannot reliably be interpreted as an absence of health conditions.

5. **The statistical association is significant but small.** The chi-square test identified a statistically significant association between work arrangement and burnout level (χ²(4) = 117.85, p < 0.001), with a small effect size (Cramer's V = 0.137).

### Managerial Implications

The findings suggest that expanding hybrid working arrangements may be worth considering as part of an employee wellbeing strategy. However, the organisation should avoid assuming that changing work arrangements alone will reduce burnout.

A cautious approach would involve piloting hybrid working options, monitoring employee burnout and social isolation, and gathering employee feedback before implementing an organisation-wide policy.

### Limitations

- The analysis uses observational survey data and cannot establish causal relationships.
- Differences in job roles, industries, and other employee characteristics may influence the observed patterns.
- Missing health information limits the interpretation of reported health measures.
- The analysis does not establish whether the survey sample represents the organisation's own workforce.
- Further organisational data and follow-up evaluation would be required before making a definitive policy decision.

### Conclusion

The analysis provides evidence of an association between work arrangement and employee burnout. It supports further investigation and a carefully evaluated hybrid-working pilot rather than an immediate organisation-wide policy change.